# 01 · Comprensión y preparación de los datos
### CRISP-DM · Fases 2 y 3 — StreamView Analytics

Este cuaderno documenta el diagnóstico de las fuentes entregadas por la organización y las
decisiones de limpieza que se derivan de él. Su propósito no es producir gráficos, sino
**establecer qué permiten y qué no permiten afirmar estos datos** antes de analizarlos.

| Fase | Contenido |
|---|---|
| 2 · Comprensión de los datos | Perfilado, calidad, diseño muestral |
| 3 · Preparación de los datos | Limpieza, armonización, integración |

In [1]:
import os
from pathlib import Path

# Permite ejecutar el cuaderno tanto desde la raiz del proyecto como desde notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

import sys
sys.path.insert(0, "src")

import pandas as pd
import numpy as np
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("Directorio de trabajo:", Path.cwd().name)

Directorio de trabajo: visualizacion


## 1. Carga de las fuentes originales

Se trabaja sobre los archivos tal como los entregó la organización, sin modificarlos.

In [2]:
peliculas = pd.read_csv("data/raw/netflix_movies_detailed_up_to_2025.csv")
series = pd.read_csv("data/raw/netflix_tv_shows_detailed_up_to_2025.csv")

print(f"Películas : {peliculas.shape[0]:,} filas x {peliculas.shape[1]} columnas")
print(f"Series    : {series.shape[0]:,} filas x {series.shape[1]} columnas")
print(f"\nColumnas solo en películas: {sorted(set(peliculas.columns) - set(series.columns))}")

Películas : 16,000 filas x 18 columnas
Series    : 16,000 filas x 16 columnas

Columnas solo en películas: ['budget', 'revenue']


## 2. Perfilado: tipos, ausencias y cardinalidad

El perfilado es el primer filtro: revela columnas inutilizables y ausencias que condicionan
el análisis posterior.

In [3]:
def perfilar(df, nombre):
    info = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "%nulos": (df.isna().sum() / len(df) * 100).round(2),
        "únicos": df.nunique(),
    })
    print(f"===== {nombre} =====")
    print(info.to_string())
    print()
    return info

_ = perfilar(peliculas, "PELÍCULAS")
_ = perfilar(series, "SERIES")

===== PELÍCULAS =====
                 tipo  nulos  %nulos  únicos
show_id         int64      0    0.00   16000
type           object      0    0.00       1
title          object      0    0.00   15485
director       object    132    0.82    9508
cast           object    204    1.27   15639
country        object    466    2.91    1463
date_added     object      0    0.00    4423
release_year    int64      0    0.00      16
rating        float64      0    0.00    2145
duration      float64  16000  100.00       0
genres         object    107    0.67    2768
language       object      0    0.00      74
description    object    132    0.82   15854
popularity    float64      0    0.00   11173
vote_count      int64      0    0.00    2693
vote_average  float64      0    0.00    2145
budget          int64      0    0.00     958
revenue         int64      0    0.00    5327

===== SERIES =====
                 tipo  nulos  %nulos  únicos
show_id         int64      0    0.00   15991
type         

### Primeras señales de alerta

- `duration` es **100% nula** en películas y toma un **único valor constante** en series.
- `director` falta en cerca de dos tercios de las series.
- `rating` y `vote_average` presentan exactamente la misma cardinalidad: hay que comprobar si
  son la misma variable duplicada.

In [4]:
print("¿rating es idéntica a vote_average?")
print("  Películas:", (peliculas["rating"] == peliculas["vote_average"]).all())
print("  Series   :", (series["rating"] == series["vote_average"]).all())

print("\nValores únicos de 'duration':")
print("  Películas:", peliculas["duration"].unique()[:5])
print("  Series   :", series["duration"].unique()[:5])

¿rating es idéntica a vote_average?
  Películas: True
  Series   : True

Valores únicos de 'duration':
  Películas: [nan]
  Series   : ['1 Seasons']


## 3. Hallazgo determinante: el diseño muestral

La distribución de títulos por año de estreno revela que **no estamos ante el catálogo completo**.

In [5]:
conteo = pd.DataFrame({
    "películas": peliculas["release_year"].value_counts().sort_index(),
    "series": series["release_year"].value_counts().sort_index(),
})
print(conteo.to_string())
print(f"\n¿Todos los años tienen el mismo número de títulos? "
      f"{conteo['películas'].nunique() == 1 and conteo['series'].nunique() == 1}")

              películas  series
release_year                   
2010               1000    1000
2011               1000    1000
2012               1000    1000
2013               1000    1000
2014               1000    1000
2015               1000    1000
2016               1000    1000
2017               1000    1000
2018               1000    1000
2019               1000    1000
2020               1000    1000
2021               1000    1000
2022               1000    1000
2023               1000    1000
2024               1000    1000
2025               1000    1000

¿Todos los años tienen el mismo número de títulos? True


> **Consecuencia metodológica.** Ambas fuentes contienen exactamente 1.000 títulos por año
> entre 2010 y 2025: se trata de una **muestra estratificada por año**, no del catálogo real.
>
> Esto invalida de antemano cualquier conclusión sobre crecimiento o contracción del catálogo
> en el tiempo, porque el volumen anual es **constante por diseño**. Todas las comparaciones
> temporales del proyecto son de composición y de calidad, nunca de volumen.

## 4. Ceros que en realidad son ausencias

Tres variables usan el cero para representar «sin dato». Promediarlas sin corregir
distorsionaría todos los resultados.

In [6]:
print("Títulos sin votos (vote_count == 0):")
print(f"  Películas: {(peliculas['vote_count'] == 0).sum():,} "
      f"({(peliculas['vote_count'] == 0).mean()*100:.1f}%)")
print(f"  Series   : {(series['vote_count'] == 0).sum():,} "
      f"({(series['vote_count'] == 0).mean()*100:.1f}%)")

print("\n¿Esos títulos figuran con calificación 0,0?")
sin_votos = series[series["vote_count"] == 0]
print(f"  De {len(sin_votos):,} series sin votos, {(sin_votos['vote_average'] == 0).sum():,} "
      f"tienen vote_average = 0")

print("\nVariables financieras en cero (solo películas):")
print(f"  budget  == 0: {(peliculas['budget'] == 0).sum():,} "
      f"({(peliculas['budget'] == 0).mean()*100:.1f}%)")
print(f"  revenue == 0: {(peliculas['revenue'] == 0).sum():,} "
      f"({(peliculas['revenue'] == 0).mean()*100:.1f}%)")

Títulos sin votos (vote_count == 0):
  Películas: 894 (5.6%)
  Series   : 3,674 (23.0%)

¿Esos títulos figuran con calificación 0,0?
  De 3,674 series sin votos, 3,661 tienen vote_average = 0

Variables financieras en cero (solo películas):
  budget  == 0: 11,153 (69.7%)
  revenue == 0: 10,355 (64.7%)


Si se promediaran los ceros como si fueran calificaciones reales, el resultado cambiaría
de forma sustantiva. La comparación lo demuestra:

In [7]:
con_ceros = series["vote_average"].mean()
sin_ceros = series.loc[series["vote_count"] > 0, "vote_average"].mean()
print(f"Calificación media de las series contando los ceros : {con_ceros:.2f}")
print(f"Calificación media excluyendo los no calificados    : {sin_ceros:.2f}")
print(f"Diferencia                                          : {sin_ceros - con_ceros:.2f} puntos")

Calificación media de las series contando los ceros : 5.42
Calificación media excluyendo los no calificados    : 7.02
Diferencia                                          : 1.61 puntos


## 5. Taxonomías de género divergentes

Al intentar comparar géneros entre formatos aparece un problema de integración: **las dos
fuentes no usan la misma clasificación**.

In [8]:
gp = set(peliculas["genres"].dropna().str.split(", ").explode())
gs = set(series["genres"].dropna().str.split(", ").explode())

print(f"Solo en películas ({len(gp - gs)}):", sorted(gp - gs))
print(f"\nSolo en series ({len(gs - gp)}):", sorted(gs - gp))
print(f"\nEn ambas ({len(gp & gs)}):", sorted(gp & gs))

Solo en películas (11): ['Action', 'Adventure', 'Fantasy', 'History', 'Horror', 'Music', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War']

Solo en series (9): ['Action & Adventure', 'Kids', 'News', 'Reality', 'Sci-Fi & Fantasy', 'Soap', 'Talk', 'Unknown', 'War & Politics']

En ambas (8): ['Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Mystery', 'Western']


> **Por qué importa.** Las películas separan `Action` y `Adventure`; las series los agrupan en
> `Action & Adventure`. Lo mismo ocurre con ciencia ficción y fantasía. Comparar géneros sin
> armonizar produciría categorías que parecen exclusivas de un formato cuando en realidad son
> un artefacto de la clasificación de origen.
>
> Este hallazgo, surgido durante la fase de análisis, obligó a **volver a la fase de preparación**
> — el comportamiento iterativo que CRISP-DM prescribe.

## 6. Otros hallazgos de calidad

In [9]:
print("Identificadores duplicados:")
print(f"  Películas: {peliculas['show_id'].duplicated().sum()}")
print(f"  Series   : {series['show_id'].duplicated().sum()}")

print("\n¿'date_added' es una fecha real de incorporación al catálogo?")
for nombre, df in [("Películas", peliculas), ("Series", series)]:
    fecha = pd.to_datetime(df["date_added"], errors="coerce")
    coincide = (fecha.dt.year == df["release_year"]).mean() * 100
    print(f"  {nombre}: coincide con release_year en el {coincide:.1f}% de los casos")

Identificadores duplicados:
  Películas: 0
  Series   : 9

¿'date_added' es una fecha real de incorporación al catálogo?
  Películas: coincide con release_year en el 100.0% de los casos
  Series: coincide con release_year en el 100.0% de los casos


> `date_added` cae **siempre** en el mismo año de estreno, por lo que no representa la fecha
> real de incorporación al catálogo y se descarta como eje temporal.

## 7. Preparación: ejecución del pipeline

Todas las decisiones anteriores están implementadas en `src/data_prep.py`. El pipeline parte de
los archivos originales y genera los conjuntos analíticos de forma reproducible.

In [10]:
import data_prep
import importlib
importlib.reload(data_prep)

data_prep.main()

peliculas_limpio.csv    16,000 filas x 22 cols
series_limpio.csv       15,991 filas x 18 cols
catalogo_unificado.csv  31,991 filas x 19 cols
catalogo_generos.csv    64,765 filas x 9 cols

Titulos calificados: 27,431 (85.7%)
Peliculas con dato financiero: 5,645


### Resumen de las decisiones aplicadas

| Problema detectado | Decisión |
|---|---|
| `rating` duplica a `vote_average` | Columna eliminada |
| `duration` nula o constante | Columna eliminada |
| Calificación 0,0 sin votos | Convertida a valor ausente |
| `budget` / `revenue` en cero | Tratados como dato faltante |
| `show_id` duplicados | Deduplicados |
| Taxonomías de género divergentes | Armonizadas a una taxonomía única en español |
| `date_added` no confiable | Descartada como eje temporal |
| Muestreo estratificado | Documentado: prohíbe conclusiones sobre volumen |

## 8. Verificación del resultado

In [11]:
catalogo = pd.read_csv("data/processed/catalogo_unificado.csv")
generos = pd.read_csv("data/processed/catalogo_generos.csv")

print(f"Catálogo unificado : {len(catalogo):,} títulos")
print(f"Tabla de géneros   : {len(generos):,} pares título-género")
print(f"Títulos calificados: {catalogo['calificado'].sum():,} "
      f"({catalogo['calificado'].mean()*100:.1f}%)")
print(f"\nGéneros armonizados ({generos['genero'].nunique()}):")
print(sorted(generos["genero"].unique()))

Catálogo unificado : 31,991 títulos
Tabla de géneros   : 64,765 pares título-género
Títulos calificados: 27,431 (85.7%)

Géneros armonizados (22):
['Acción y Aventura', 'Animación', 'Bélico y Político', 'Ciencia Ficción y Fantasía', 'Comedia', 'Crimen', 'Documental', 'Drama', 'Familiar', 'Histórico', 'Infantil', 'Misterio', 'Musical', 'Noticias', 'Película para TV', 'Reality', 'Romance', 'Suspenso', 'Talk Show', 'Telenovela', 'Terror', 'Western']


---
**Siguiente paso:** `02_analisis_exploratorio.ipynb` — CRISP-DM fase 4.